# Stock Return Prediction Using Machine Learning
This project uses historical A-share stock data to engineer financial features and train machine learning models to predict next-day price direction (up or down).

In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# ── 1. Retrieve data ─────────────────────────────────────────────────────────
symbol = "000001"
df = ak.stock_zh_a_hist(symbol=symbol, period="daily",
                        start_date="20200101", end_date="20241231",
                        adjust="qfq")

df = df.rename(columns={"日期": "date", "开盘": "open", "收盘": "close",
                         "最高": "high", "最低": "low", "成交量": "volume"})
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()[["open", "close", "high", "low", "volume"]]

print(f"Data loaded: {len(df)} trading days")
df.head()

In [ ]:
# ── 2. Feature engineering ───────────────────────────────────────────────────
# Daily return
df["return"] = df["close"].pct_change()

# Moving averages
df["MA5"]  = df["close"].rolling(5).mean()
df["MA20"] = df["close"].rolling(20).mean()

# Momentum: return over past 5 days
df["momentum"] = df["close"].pct_change(5)

# Volatility: 10-day rolling std of returns
df["volatility"] = df["return"].rolling(10).std()

# Price relative to MA20
df["price_to_MA20"] = df["close"] / df["MA20"]

# Volume change
df["volume_change"] = df["volume"].pct_change()

# Target: 1 if next day's close is higher, 0 otherwise
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)

# Drop rows with NaN
df = df.dropna()

print(f"Features ready. Samples: {len(df)}")
print(f"Target distribution:\n{df['target'].value_counts()}")
df.tail(5)

In [ ]:
# ── 3. Train / test split ────────────────────────────────────────────────────
features = ["return", "MA5", "MA20", "momentum", "volatility",
            "price_to_MA20", "volume_change"]

X = df[features]
y = df["target"]

# Time-series split: no shuffle to avoid look-ahead bias
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

In [ ]:
# ── 4. Train models ──────────────────────────────────────────────────────────
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)

print("── Random Forest ────────────────────────")
print(f"  Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test, rf_pred, target_names=["Down", "Up"]))

print("── Logistic Regression ──────────────────")
print(f"  Accuracy: {accuracy_score(y_test, lr_pred):.3f}")
print(classification_report(y_test, lr_pred, target_names=["Down", "Up"]))

In [ ]:
# ── 5. Visualise results ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Stock Return Prediction — Model Evaluation", fontsize=13)

# Confusion matrices
for ax, pred, title in zip(axes[:2],
                           [rf_pred, lr_pred],
                           ["Random Forest", "Logistic Regression"]):
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Down", "Up"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Down", "Up"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)

# Feature importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
importances.plot(kind="barh", ax=axes[2], color="steelblue")
axes[2].set_title("Feature Importance (RF)")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("ml_prediction_results.png", dpi=150)
plt.show()
print("Chart saved as ml_prediction_results.png")

In [ ]:
# ── 6. Summary ───────────────────────────────────────────────────────────────
print("── Summary ──────────────────────────────────────────")
print(f"  Stock:                    {symbol}")
print(f"  Period:                   {df.index[0].date()} to {df.index[-1].date()}")
print(f"  Features used:            {len(features)}")
print(f"  Training samples:         {len(X_train)}")
print(f"  Test samples:             {len(X_test)}")
print(f"  Random Forest accuracy:   {accuracy_score(y_test, rf_pred):.3f}")
print(f"  Logistic Reg. accuracy:   {accuracy_score(y_test, lr_pred):.3f}")
top_feature = importances.idxmax()
print(f"  Most important feature:   {top_feature}")